# 🎨 Oddity AI — Colab GPU Backend

This notebook runs your Oddity server on Google Colab's free GPU.
Your local Photoshop plugin connects to it via an **ngrok tunnel**.

## How to use:
1. **Run Cell 1** → Installs everything + clones your repo
2. **Run Cell 2** → Paste your ngrok token and start the server
3. **Copy the URL** printed at the end → Paste into your plugin's Connection Settings

> ⚠️ Make sure to select **GPU runtime**: `Runtime → Change runtime type → T4 GPU`

---
## Cell 1: Install Dependencies & Clone Repo
This installs PyTorch (CUDA), diffusers, FastAPI, and all other requirements, then clones your project from GitHub.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies & Clone Repository
# ============================================================

import subprocess, os

print("📦 Installing PyTorch with CUDA support...")
subprocess.run(["pip", "install", "-q", "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

print("📦 Installing diffusers, transformers, and ML libraries...")
subprocess.run(["pip", "install", "-q",
                "diffusers>=0.31.0", "transformers>=4.36.0", "accelerate>=0.25.0",
                "safetensors>=0.4.0", "sentencepiece>=0.1.99", "protobuf>=4.25.0",
                "huggingface_hub>=0.20.0"], check=True)

print("📦 Installing FastAPI, uvicorn, and server dependencies...")
subprocess.run(["pip", "install", "-q",
                "fastapi>=0.104.0", "uvicorn[standard]>=0.24.0",
                "pillow>=10.0.0", "python-multipart>=0.0.6",
                "pyngrok"], check=True)

# Clone the repo (or pull latest if already cloned)
REPO_URL = "https://github.com/Archi-Ezzat/Oddity.git"
REPO_DIR = "/content/Oddity"

if os.path.exists(REPO_DIR):
    print("🔄 Repository already exists, pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print("📥 Cloning Oddity repository from GitHub...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Create model directory structure
for d in [
    "models/flux/checkpoints", "models/sdxl/checkpoints",
    "models/sd3/checkpoints", "models/sd15/checkpoints",
    "components/shared/clip", "components/shared/vae",
    "components/flux/vae", "components/sdxl/vae",
    "components/sd3/vae", "components/sd15/vae"
]:
    os.makedirs(f"/content/{d}", exist_ok=True)

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
    print(f"\n✅ GPU detected: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("\n⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

print("\n✅ All dependencies installed and repo cloned!")
print("👉 Now run Cell 2 below.")

---
## Cell 2: Start Server with ngrok Tunnel

1. Get your **ngrok auth token** from: https://dashboard.ngrok.com/get-started/your-authtoken
2. Paste it in the `NGROK_AUTH_TOKEN` field below
3. Run this cell — it will print your public URL
4. Copy that URL into your Photoshop plugin's **Connection Settings** (gear icon)

In [ ]:
# ============================================================
# CELL 2: Start ngrok Tunnel + Oddity Server
# ============================================================

# ⬇️⬇️⬇️ PASTE YOUR NGROK TOKEN HERE ⬇️⬇️⬇️
NGROK_AUTH_TOKEN = ""  # <-- Paste your token between the quotes
# ⬆️⬆️⬆️ PASTE YOUR NGROK TOKEN HERE ⬆️⬆️⬆️

# ----------------------------------------------------------

import os, sys, threading, time

if not NGROK_AUTH_TOKEN:
    print("❌ You need to paste your ngrok auth token above!")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
    raise ValueError("NGROK_AUTH_TOKEN is empty")

# Set environment variables for the Oddity server
os.environ["ODDITY_HOST"] = "0.0.0.0"
os.environ["ODDITY_PORT"] = "5000"
os.environ["ODDITY_MODELS_DIR"] = "/content/models"
os.environ["ODDITY_COMPONENTS_DIR"] = "/content/components"

# Setup ngrok tunnel
from pyngrok import ngrok, conf

# Kill any existing tunnels
ngrok.kill()

# Authenticate
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open tunnel to port 5000
public_url = ngrok.connect(5000, "http")
public_url_str = str(public_url).replace('NgrokTunnel: ', '').split(' ')[0]
# Clean up the URL — extract just the https:// part
if '"' in public_url_str:
    public_url_str = public_url_str.strip('"')

print("")
print("=" * 60)
print("  🚀 ODDITY SERVER IS STARTING")
print("=" * 60)
print(f"")
print(f"  📋 Your public URL:")
print(f"")
print(f"     {public_url}")
print(f"")
print(f"  👉 Copy this URL and paste it into your Photoshop")
print(f"     plugin's Connection Settings (gear icon).")
print(f"     Then click Connect.")
print(f"")
print("=" * 60)
print("")
print("⏳ Starting FastAPI server... (this cell will keep running)")
print("   The server is ready when you see 'Uvicorn running on...'")
print("")

# Change to the backend directory and run the server
os.chdir("/content/Oddity/backend")
sys.path.insert(0, "/content/Oddity/backend")

# Run the server (this blocks — keep this cell running!)
import uvicorn
from server import app

uvicorn.run(app, host="0.0.0.0", port=5000, log_level="info")

---
## 🛑 Troubleshooting

| Problem | Solution |
|---------|----------|
| `No GPU detected` | Go to `Runtime → Change runtime type → T4 GPU` |
| `ngrok connection failed` | Make sure your token is correct, and you don't have another tunnel running |
| Plugin says "OFFLINE" | Check that Cell 2 is still running and the URL is correct |
| Model download is slow | Colab's internet is fast — large models (24GB) still take a few minutes |
| `CUDA out of memory` | Use smaller models (SD 1.5 or SDXL) instead of Flux |

### Free Colab GPU Limits
- **T4 GPU**: 15 GB VRAM — enough for SDXL and SD 1.5 models
- **Session length**: ~12 hours max, then it disconnects
- **Flux models**: Need 12+ GB VRAM, works on T4 but tight
- **SDXL models**: 6-7 GB, run comfortably on T4
- **SD 1.5 models**: 2-4 GB, very fast on T4